# Intent Model Training Visualization (Colab)

Use this notebook to train DistilBERT for intent routing and generate visual proof of training behavior (loss/accuracy curves).

In [ ]:
!pip -q install transformers datasets accelerate torch pandas matplotlib seaborn

In [ ]:
from pathlib import Path
import subprocess
import json
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root() -> Path:
    candidates = [
        Path('/content/StylesenseSL'),
        Path('/content'),
        Path.cwd(),
    ]
    for c in candidates:
        if (c / 'backend' / 'src' / 'services' / 'agentic_ai').exists():
            return c
    raise RuntimeError('Could not find repo root. Clone repo under /content first.')

REPO_ROOT = find_repo_root()
BACKEND_ROOT = REPO_ROOT / 'backend'
MODEL_DIR = BACKEND_ROOT / 'src' / 'services' / 'agentic_ai' / 'agents' / 'models' / 'intent_distilbert'
DATA_CSV = BACKEND_ROOT / 'src' / 'services' / 'agentic_ai' / 'data' / 'intent' / 'intent_dataset_8400.csv'

print('REPO_ROOT:', REPO_ROOT)
print('BACKEND_ROOT:', BACKEND_ROOT)
print('MODEL_DIR:', MODEL_DIR)
print('DATA_CSV:', DATA_CSV)

In [ ]:
if not DATA_CSV.exists():
    cmd = [
        'python', '-m', 'src.services.agentic_ai.scripts.generate_intent_dataset_csv',
        '--output', str(DATA_CSV.relative_to(BACKEND_ROOT).as_posix()),
        '--rows', '8400'
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=BACKEND_ROOT, check=True)

print('Dataset exists:', DATA_CSV.exists())

In [ ]:
cmd = [
    'python', '-m', 'src.services.agentic_ai.scripts.train_intent_distilbert',
    '--data', str(DATA_CSV.relative_to(BACKEND_ROOT).as_posix()),
    '--output', str(MODEL_DIR.relative_to(BACKEND_ROOT).as_posix()),
    '--epochs', '3',
    '--batch-size', '16'
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=BACKEND_ROOT, check=True)

In [ ]:
state_path = MODEL_DIR / 'training_runs' / 'checkpoint-420' / 'trainer_state.json'
if not state_path.exists():
    checkpoints = sorted((MODEL_DIR / 'training_runs').glob('checkpoint-*'))
    if checkpoints:
        state_path = checkpoints[-1] / 'trainer_state.json'

if not state_path.exists():
    raise FileNotFoundError(f'trainer_state.json not found under {MODEL_DIR / 
}')

state = json.loads(state_path.read_text(encoding='utf-8'))
log_history = state.get('log_history', [])
df = pd.DataFrame(log_history)
df.head()

In [ ]:
plot_df = df.copy()
if 'epoch' not in plot_df.columns:
    plot_df['epoch'] = range(1, len(plot_df) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'loss' in plot_df.columns:
    axes[0].plot(plot_df['epoch'], plot_df['loss'], marker='o', label='train_loss')
if 'eval_loss' in plot_df.columns:
    eval_df = plot_df.dropna(subset=['eval_loss'])
    axes[0].plot(eval_df['epoch'], eval_df['eval_loss'], marker='s', label='eval_loss')
axes[0].set_title('Training vs Evaluation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

if 'eval_accuracy' in plot_df.columns:
    eval_df = plot_df.dropna(subset=['eval_accuracy'])
    axes[1].plot(eval_df['epoch'], eval_df['eval_accuracy'], marker='o', color='green')
axes[1].set_title('Evaluation Accuracy by Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
cfg_path = MODEL_DIR / 'intent_inference_config.json'
cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
proof = {
    'val_accuracy': cfg.get('val_accuracy'),
    'accepted_accuracy': cfg.get('accepted_accuracy'),
    'confidence_threshold': cfg.get('confidence_threshold'),
    'temperature': cfg.get('temperature'),
    'dataset_rows': cfg.get('dataset_rows'),
}
print(json.dumps(proof, indent=2))